## Objective
In this project, my primary focus is on addressing insights related to Fleet Management and Supply Chain. To achieve this goal efficiently, the project has been divided into few objectives:

# Data Reading

In [1]:
# import the libraries
import pandas as pd
import re
from IPython.display import display, Markdown

There are excel workbooks so i had to extract the sheets of them and convert to csv so I can work on easly

In [14]:
excelPath="DimensionTables.xlsx"
excelFile = pd.ExcelFile(excelPath)
print(excelFile.sheet_names)

['Drivers', 'Vehicles', 'Customers']


In [15]:
for name in excelFile.sheet_names:
    df = pd.read_excel(excelPath, sheet_name=name)
    df.to_csv(f"{name}.csv", index=False, encoding="utf-8-sig")

In [9]:
excelPath2="fCosts.xlsx"
excelFile2 = pd.ExcelFile(excelPath2)
print(excelFile2.sheet_names)

['fKMTraveled']


In [ ]:
sheetName = excelFile2.sheet_names[0]
df = pd.read_excel(excelPath2, sheet_name=sheetName)
df.to_csv(f"{sheetName}.csv", index=False, encoding="utf-8-sig")

In [10]:
excelPath3="Dim States.xlsx"
excelFile3 = pd.ExcelFile(excelPath3)
print(excelFile3.sheet_names)

['STATES']


In [ ]:
sheetName = excelFile3.sheet_names[0]
df = pd.read_excel(excelPath3, sheet_name=sheetName)
df.to_csv(f"{sheetName}.csv", index=False, encoding="utf-8-sig")

In [63]:
# ===Reading Files===
F_Freight=pd.read_csv('fFreight.csv')
F_KMTraveled=pd.read_csv('fKMTraveled.csv')
D_Customers=pd.read_csv('Customers.csv')
D_Drivers=pd.read_csv('Drivers.csv')
D_STATES=pd.read_csv('STATES.csv')
D_Vehicles=pd.read_csv('Vehicles.csv')
tables_list = [
    ("F_Freight", F_Freight),
    ("F_KMTraveled", F_KMTraveled),
    ("D_Customers", D_Customers),
    ("D_Drivers", D_Drivers),
    ("D_STATES", D_STATES),
    ("D_Vehicles", D_Vehicles)
]

# Data Exploration

In [64]:
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.width', 1500)

def auditData(df, fileName="Dataset"):
    display(Markdown(f"# 📊 Audit Report — `{fileName}`"))

    # Preview
    display(Markdown("## 👀 Preview"))
    display(df.head())

    # Basic Audit
    display(Markdown("## 🧱 Basic Audit"))

    # Schema + Missing Values
    schemaDf = pd.DataFrame({
        "Column": df.columns,
        "Non-Null Count": df.notna().sum().values,
        "Dtype": df.dtypes.values,
        "Missing Count": df.isna().sum().values
    })
    display(Markdown("### Schema & Missing Values"))
    display(schemaDf)

    # Duplicated Rows + Dataset Summary
    totalRows = len(df)
    totalCols = df.shape[1]
    dtypesSummary = ', '.join([f"{dt} {cnt}" for dt, cnt in df.dtypes.value_counts().items()])
    dupSummaryDf = pd.DataFrame({
        "Duplicated Rows": [df.duplicated().sum()],
        "Total Rows": [totalRows],
        "Total Columns": [totalCols],
        "Dtypes Summary": [dtypesSummary]
    })
    display(Markdown("### Duplicates & Dataset Summary"))
    display(dupSummaryDf)

    # Categorical Audit
    catCols = df.select_dtypes(include=['object', 'category', 'string']).columns
    if len(catCols):
        display(Markdown("## 🔤 Categorical Audit"))

        # Unique counts table
        uniqueCounts = {col: df[col].nunique(dropna=False) for col in catCols}
        display(Markdown("### Unique Counts per Column"))
        display(pd.DataFrame(uniqueCounts, index=["Unique Count"]).T)

        # Full unique values (wide display dynamic)
        for col in catCols:
            if not re.search(r'id|date|key|time', col, re.IGNORECASE):
                display(Markdown(f"### `{col}` — All Unique Values"))
                values = df[col].astype(str).unique()
                n_values = len(values)

                # Dynamic columns based on number of values
                ncols = max(5, int(n_values**0.5))  
                nrows = (n_values + ncols - 1) // ncols
                wideArr = [values[i*ncols:(i+1)*ncols] for i in range(nrows)]
                wideDf = pd.DataFrame(wideArr)
                display(wideDf.fillna(""))

                # Check spaces
                spaceDf = df.assign(
                    leadTrail=df[col].astype(str).str.len()
                              != df[col].astype(str).str.strip().str.len(),
                    doubleSpace=df[col].astype(str).str.contains(r'\s{2,}', regex=True)
                ).loc[
                    lambda x: x['leadTrail'] | x['doubleSpace'],
                    [col, 'leadTrail', 'doubleSpace']
                ].drop_duplicates()
                if not spaceDf.empty:
                    display(Markdown("⚠️ Space Issues"))
                    display(spaceDf)

    # Numerical Audit (Invalid + Quantiles)
    numCols = df.select_dtypes(include=['number']).columns
    if len(numCols):
        display(Markdown("## 🔢 Numerical Audit"))
        display(Markdown("### Statistics"))
        display(df[numCols].describe())

        # Prepare combined table: Invalid + quantiles
        combinedDict = {}
        for col in numCols:
            combinedDict[col] = [(df[col] <= 0).sum()] + df[col].quantile([0,0.01,0.25,0.5,0.75,0.99,1]).tolist()
        indexLabels = ["Invalid (<=0)", 0, 0.01, 0.25, 0.5, 0.75, 0.99, 1]
        combinedDf = pd.DataFrame(combinedDict, index=indexLabels)
        display(Markdown("### Invalid Values ⬅️➡️ Quantiles"))
        display(combinedDf)

    # Date & Time Audit 
    dateCols = list(set(
        df.select_dtypes(include=['datetime', 'datetimetz']).columns.tolist() +
        [c for c in df.columns if re.search(r'year|month|day|date|time', c, re.IGNORECASE)]
    ))
    if dateCols:
        display(Markdown("## 📅 Date & Time Audit"))
        for col in dateCols:
            if df[col].dtype in ['object', 'string', 'category']:
                tmp = pd.to_datetime(df[col], errors='coerce')
                if tmp.notna().any():
                    display(pd.DataFrame({
                        "column": [col],
                        "min": [tmp.min()],
                        "max": [tmp.max()]
                    }))
                else:
                    display(pd.DataFrame({
                        "column": [col],
                        "status": ["not date-parsable"]
                    }))
            else:
                # Numeric column: show min/max as is
                display(pd.DataFrame({
                    "column": [col],
                    "min": [df[col].min()],
                    "max": [df[col].max()]
                }))

# === Function to audit a list of DataFrames ===
def auditDataFrames(dfs_list):
    for name, df in dfs_list:
        auditData(df, name)


auditDataFrames(tables_list)

# 📊 Audit Report — `F_Freight`

## 👀 Preview

,Date,Customer ID,Truck ID,Invoice Number,Freight ID,City,Net Revenue,Weight (Kg),Weight (Cubic),Goods Value
0,2018/01/02,10975,38,774571,02/01/2018:MMA-5946,Herrings Crossroads,7.42,2.50,3.0,247.39
1,2018/01/02,22346,23,774507,02/01/2018:MMA-4836,Enders,3.54,2.62,3.0,118.05
2,2018/01/02,12208,23,774516,02/01/2018:MMA-4836,Shavertown,8.98,5.54,9.0,299.01
3,2018/01/02,1929,23,774626,02/01/2018:MMA-4836,Enders,12.49,7.80,9.0,416.27
4,2018/01/02,6198,23,774623,02/01/2018:MMA-4836,Enders,2.49,8.55,9.0,82.90


## 🧱 Basic Audit

### Schema & Missing Values

...

,Truck ID,Year
count,31.000000,31.000000
mean,25.709677,2012.129032
std,13.249386,3.490386
min,2.000000,2006.000000
25%,18.500000,2009.000000
50%,27.000000,2013.000000
75%,36.500000,2014.000000
max,45.000000,2019.000000


### Invalid Values ⬅️➡️ Quantiles

,Truck ID,Year
Invalid (<=0),0.0,0.0
0,2.0,2006.0
0.01,2.3,2006.6
0.25,18.5,2009.0
0.5,27.0,2013.0
0.75,36.5,2014.0
0.99,44.7,2019.0
1,45.0,2019.0


## 📅 Date & Time Audit

,column,min,max
0,Year,2006,2019


# Data Summary

The dataset provides many data tables including freights and their costs , customers and their states, drivers and their vehicles. After examining the data fields, I noticed that the dataset generally represents the following key information

* Customer: General information about customers including identifiers and addresses

* drivers: Information about drivers including identifiers and names

* vehicles: Vehicles information including Plate, brand, year of manufacturing and truck&trailer types

* Freights: Every frieght with information about costs, revenue, kms, weights, truck used and the city it went to

We found invalid values here: weights are zero however revenue and goods value are not

In [65]:
F_Freight[F_Freight['Weight (Kg)']==0]

,Date,Customer ID,Truck ID,Invoice Number,Freight ID,City,Net Revenue,Weight (Kg),Weight (Cubic),Goods Value
19238,2018/06/04,3250,19,841074,04/06/2018:MDB-1602,Minnewawa,63.86,0.0,0.0,884.40
26386,2018/08/02,31836,19,867643,02/08/2018:MDB-1602,Mineola,63.86,0.0,0.0,810.94
76344,2019/06/13,1600,22,1010816,13/06/2019:MMA-5926,Sowash,93.98,0.0,0.0,2100.00


I think those days are for meintenance only

In [66]:
F_KMTraveled[F_KMTraveled['KM Traveled']==0]

,Date,Truck ID,Drive ID,KM Traveled,Liters,Fuel,Maintenance,Fixed Costs
17,2018-01-01,34,1,0,0.0,0.0,929.45,7212.140655
123,2018-07-01,5,1,0,0.0,0.0,1140.86,7293.480138
142,2018-08-01,37,26,0,0.0,0.0,488.77,6117.121100


We found half of data geting weight (kg) = weight(Cubic)

so i think cubic is only duplicate of the kg so we won't depend on this column

In [67]:
mask = (F_Freight['Weight (Kg)'] == F_Freight['Weight (Cubic)'])
f"{mask.mean() * 100:.2f}%"

'49.63%'

We found some big values in data so we checked them

In [ ]:
F_Freight[F_Freight['Goods Value']>450000]
''' 
Goods Value is 4,987.6 for each kg, that is very big 
Maybe They are expensive Electronics
'''

,Date,Customer ID,Truck ID,Invoice Number,Freight ID,City,Net Revenue,Weight (Kg),Weight (Cubic),Goods Value
67679,2019/05/02,22719,23,990576,02/05/2019:MMA-4836,Enders,17301.65,192.1,192.1,957357.99


In [ ]:
F_Freight[F_Freight['Weight (Kg)']>30000]
''' 
Goods Value is 0.038 for each kg, that is very small 
Maybe They are Inexpensive raw materials: Sand, gravel, cement, non-precious metal ores
'''

,Date,Customer ID,Truck ID,Invoice Number,Freight ID,City,Net Revenue,Weight (Kg),Weight (Cubic),Goods Value
13815,2018/04/16,169,33,821503,16/04/2018:MKU-5802,Spring City,15492.95,31748.0,31748.0,1206.7


We also found date columns to be object type so we have to change it to datetime

# Data Transformation and Cleaning

We need to do type casting for date column from object to datetime 

In [71]:
F_KMTraveled['Date'] = pd.to_datetime(F_KMTraveled['Date'], errors='coerce')
F_Freight['Date'] = pd.to_datetime(F_Freight['Date'], errors='coerce')

Since values that have weights zero are only 3 and that is very small percentage of total, we will drop them

In [72]:
mask = (F_Freight['Weight (Kg)'] == 0) & (F_Freight['Weight (Cubic)'] == 0)
F_Freight = F_Freight.loc[~mask]

# Feature Metrics
The feature metrics that planning to create in Power BI which helps to further analyze Drivers, Customers, Revenue and many more.

### Total Costs

Total Costs is a significant financial metric in supply chain management that calculates the overall cost associated with Maintenance, Fuel, Fixed Costs.

The formula for calculating Total Inventory Cost is: Sum of`Maintenance cost` + Sum of`Fuel Cost` + Sum of`Fixed Cost`.

#### Shipment Delay in Days
Shipment Delay  = [Shipment Days - Actual] - ['Shipment Days - Scheduled']

### Net Profit and Profit Margin

Profit Margin helps to assessing the profitability of the supply chain operations. It provides insights into the effectiveness of cost control and pricing strategies, enabling organizations to make adjustments to enhance overall profitability.

Profit Margin : Net Profit / Total Revenue *100

### Drivers Efficiency

`Avg Fuel consumption per km`, `Avg Fuel efficiency`, and `Avg Fuel cost per km` for each driver

Avg Fuel efficiency  = Total KMs traveled / Literes Consumed

Avg Fuel consumption per km = Literes Consumed / Total KMs traveled

Avg Fuel cost per km = Fuel Cost / Total KMs traveled


### Customer Retention

Customer Retention helps to find which Customers who return to make multiple orders

% Customer with multiple orders = repeated customers / active customers

# Data Exporting & Conclusion

## Exporting Cleaned Data

Save the cleaned and processed DataFrames to a csv file for further visualization and analysis in Power BI.

In [73]:
# Export DataFrames to CSV
F_Freight.to_csv('F_Freight.csv', index=False)
F_KMTraveled.to_csv('F_KMTraveled.csv', index=False)
D_Customers.to_csv('D_Customers.csv', index=False)
D_Drivers.to_csv('D_Drivers.csv', index=False)
D_STATES.to_csv('D_STATES.csv', index=False)
D_Vehicles.to_csv('D_Vehicles.csv', index=False)

## Conclusion

Through this analysis, we successfully explored and performed data cleaning on the `F_Freight` and `F_KMTraveled` DataFrames. Important metrics and Kpis such as `Total Revenue`, `Total Costs` `Net Proft` and their %YoY vs PY

 `Avg Fuel consumption per km`, `Avg Fuel efficiency`, and `Avg Fuel cost per km` for each driver will be calculated in Power BI to understand Fleet Management performance and supply chain efficiency for the company.

Additionally, we removed some anomalous data points that could impact the analysis results. The cleaned and processed data will be exported to a csv file for further analysis and visualization in Tableau.


## Closing

As an aspiring data engineer/data analyst, I will consistently look for opportunities to improve my skills and insights.Thank you so much.

This analysis was conducted on Khnown Data Set by Kaggle, with the aim of demonstrating data analytics expertise and providing actionable insights to address real-world business challenges. Throughout this process, I strive to use effective data processing, data cleaning, and advanced analytics techniques to generate meaningful conclusions and support decision making.